# 1. Analysis - Income Statement

## Notebook Summary

This notebook analyzes a single companyâ€™s income statement history from Financial Modeling Prep using the local `TICKER` parameter near the top of this notebook. It is organized into data loading and preparation, summary snapshots, core trend charts, seasonal growth views, margin-driver diagnostics, and revenue segmentation where FMP data is available.

Run the notebook from top to bottom after updating the local `TICKER` parameter near the top of this notebook so the helper functions, annual and quarterly datasets, and all downstream charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly income statement history, and calculate the derived growth, margin, and ratio fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Setup
from pathlib import Path
import sys

import pandas as pd

financial_statement_params = {
    "ticker_str": "SOXL",
    "annual_limit": 20,
    "quarterly_limit": 40,
}

TICKER = financial_statement_params["ticker_str"]
ANNUAL_LIMIT = financial_statement_params["annual_limit"]
QUARTERLY_LIMIT = financial_statement_params["quarterly_limit"]

# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.data import get_income_statement_history


TICKER

In [ ]:
# 3. Load income statement history through Quantapp.data
income_statement_history = get_income_statement_history(
    TICKER,
    annual_limit=ANNUAL_LIMIT,
    quarterly_limit=QUARTERLY_LIMIT,
)

SYMBOL = income_statement_history.symbol
company_name = income_statement_history.company_name
chart_label = income_statement_history.chart_label
annual_revenue = income_statement_history.annual
quarterly_revenue = income_statement_history.quarterly
annual_product_segmentation = income_statement_history.annual_product_segmentation
annual_product_segments = income_statement_history.annual_product_segments
annual_geographic_segmentation = income_statement_history.annual_geographic_segmentation
annual_geographic_segments = income_statement_history.annual_geographic_segments
available_revenue_segmentations = income_statement_history.available_revenue_segmentations

annual_revenue.tail()

In [ ]:
# 4. Review quarterly income statement history
quarterly_revenue.tail()

In [ ]:
# 5. Build annual summary stats
latest_annual = annual_revenue.iloc[-1]
annual_years = max(len(annual_revenue) - 1, 1)
annual_cagr = (latest_annual["revenue"] / annual_revenue.iloc[0]["revenue"]) ** (1 / annual_years) - 1 if len(annual_revenue) > 1 else pd.NA

annual_summary = pd.DataFrame(
    [
        {
            "metric": "Latest annual revenue",
            "value": latest_annual["revenue"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual revenue CAGR",
            "value": annual_cagr,
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual gross margin",
            "value": latest_annual["grossMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual operating margin",
            "value": latest_annual["operatingMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual net margin",
            "value": latest_annual["netMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual gross margin change",
            "value": latest_annual["grossMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual operating margin change",
            "value": latest_annual["operatingMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual net margin change",
            "value": latest_annual["netMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
    ]
)

In [ ]:
# 6. Build quarterly summary stats
latest_quarter = quarterly_revenue.iloc[-1]
latest_ttm = quarterly_revenue.dropna(subset=["ttmRevenue"]).iloc[-1]

quarterly_summary = pd.DataFrame(
    [
        {
            "metric": "Latest quarterly revenue",
            "value": latest_quarter["revenue"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest TTM revenue",
            "value": latest_ttm["ttmRevenue"],
            "asOf": latest_ttm["date"].date(),
        },
        {
            "metric": "Latest quarterly YoY growth",
            "value": latest_quarter["revenueYoY"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly gross margin",
            "value": latest_quarter["grossMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly operating margin",
            "value": latest_quarter["operatingMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly net margin",
            "value": latest_quarter["netMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly gross margin change",
            "value": latest_quarter["grossMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly operating margin change",
            "value": latest_quarter["operatingMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly net margin change",
            "value": latest_quarter["netMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
    ]
)

## Trend and Seasonality Charts

These cells move from summary tables into visualization. They cover annual trend charts, quarterly trend charts, and the seasonal quarterly growth view so you can separate long-term operating direction from quarter-specific patterns.

In [ ]:
# 7. Plot annual income statement trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_income_statement_trends

annual_fig = plot_income_statement_trends(
    annual_revenue,
    chart_label=chart_label,
    period_label="annual",
)
annual_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly income statement trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_income_statement_trends

quarterly_fig = plot_income_statement_trends(
    quarterly_revenue,
    chart_label=chart_label,
    period_label="quarterly",
)
quarterly_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 9. Plot seasonal quarterly growth rates
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_seasonal_growth_rates

seasonal_growth_fig = plot_seasonal_growth_rates(quarterly_revenue, chart_label=chart_label)
seasonal_growth_fig.show(config={"responsive": True, "displaylogo": False})

## Common-Size, EPS, and TTM Profitability

These cells recast the income statement as a share of revenue, isolate per-share performance versus dilution, and track rolling four-quarter profit conversion before moving into the margin-driver charts.

In [ ]:
# 10. Build common-size income statement views
from IPython.display import display

def build_common_size_statement(frame: pd.DataFrame, column_labels: pd.Series) -> pd.DataFrame:
    statement_rows = [
        ("Revenue", pd.Series(1.0, index=frame.index)),
        ("Cost of revenue", -frame["costOfRevenuePctRevenue"]),
        ("Gross profit", frame["grossProfitPctRevenue"]),
        ("Operating expenses", -frame["operatingExpensesPctRevenue"]),
    ]

    optional_rows = [
        ("G&A within opex", -frame["gaPctRevenue"]),
        ("SG&A within opex", -frame["sgaPctRevenue"]),
        ("Marketing within SG&A", -frame["marketingPctRevenue"]),
        ("R&D within opex", -frame["rdPctRevenue"]),
    ]
    for label, values in optional_rows:
        if values.fillna(0).abs().gt(1e-6).any():
            statement_rows.append((label, values))

    statement_rows.extend(
        [
            ("Operating income", frame["operatingMargin"]),
            ("Other income / expense", frame["otherIncomeExpensePctRevenue"]),
            ("Pre-tax income", frame["incomeBeforeTaxMargin"]),
            ("Income tax expense", -frame["incomeTaxExpensePctRevenue"]),
            ("Net income", frame["netMargin"]),
        ]
    )

    statement = pd.DataFrame(
        [values.reset_index(drop=True).tolist() for _, values in statement_rows],
        index=[label for label, _ in statement_rows],
        columns=column_labels.reset_index(drop=True).tolist(),
    )
    return statement.dropna(how="all", axis=0)

annual_common_size_frame = annual_revenue.tail(min(len(annual_revenue), 6)).copy()
annual_common_size_labels = annual_common_size_frame["calendarYear"].astype("Int64").astype(str)
annual_common_size = build_common_size_statement(annual_common_size_frame, annual_common_size_labels)

quarterly_common_size_frame = quarterly_revenue.tail(min(len(quarterly_revenue), 8)).copy()
quarterly_common_size_labels = (
    quarterly_common_size_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_common_size_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_common_size_frame["date"].dt.strftime("%b %Y")
    + ")"
    )
quarterly_common_size = build_common_size_statement(quarterly_common_size_frame, quarterly_common_size_labels)

display(
    annual_common_size.style.format("{:.1%}").set_caption(
        f"{SYMBOL} annual common-size income statement (% of revenue)"
    )
)
display(
    quarterly_common_size.style.format("{:.1%}").set_caption(
        f"{SYMBOL} quarterly common-size income statement (% of revenue)"
    )
)

In [ ]:
# 11. Analyze EPS and dilution
from IPython.display import display

annual_eps_frame = annual_revenue.tail(min(len(annual_revenue), 8)).copy()
quarterly_eps_frame = quarterly_revenue.tail(min(len(quarterly_revenue), 12)).copy()
analysis_label = locals().get("chart_label", locals().get("company_name", SYMBOL))

def format_eps_summary_value(metric: str, value: float) -> str:
    if pd.isna(value):
        return "-"
    if "dilution" in metric.lower() or "growth" in metric.lower():
        return f"{value:.2%}"
    return f"{value:,.2f}"

eps_dilution_summary = pd.DataFrame(
    [
        {
            "metric": "Latest annual basic EPS",
            "value": annual_revenue.iloc[-1]["eps"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest annual diluted EPS",
            "value": annual_revenue.iloc[-1]["epsDiluted"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest annual dilution",
            "value": annual_revenue.iloc[-1]["dilutionPct"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly basic EPS",
            "value": quarterly_revenue.iloc[-1]["eps"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly diluted EPS",
            "value": quarterly_revenue.iloc[-1]["epsDiluted"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly dilution",
            "value": quarterly_revenue.iloc[-1]["dilutionPct"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Quarterly diluted share YoY growth",
            "value": quarterly_revenue.iloc[-1]["dilutedSharesYoY"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
    ]
)
eps_dilution_summary_display = eps_dilution_summary.copy()
eps_dilution_summary_display["value"] = [
    format_eps_summary_value(metric, value)
    for metric, value in zip(eps_dilution_summary_display["metric"], eps_dilution_summary_display["value"])
]
display(eps_dilution_summary_display)

from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_eps_dilution

eps_dilution_fig = plot_eps_dilution(
    annual_eps_frame,
    quarterly_eps_frame,
    analysis_label=analysis_label,
)
eps_dilution_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 12. Track TTM profit conversion
from IPython.display import display

ttm_frame = quarterly_revenue.dropna(subset=["ttmRevenue"]).copy()
analysis_label = locals().get("chart_label", locals().get("company_name", SYMBOL))

if ttm_frame.empty:
    raise RuntimeError(f"Need at least four quarters of history to compute TTM profit metrics for {SYMBOL}.")

def format_ttm_summary_value(metric: str, value: float) -> str:
    if pd.isna(value):
        return "-"
    if "margin" in metric.lower():
        return f"{value:.2%}"
    return f"{value:,.0f}"

latest_ttm_metrics = ttm_frame.iloc[-1]
ttm_summary = pd.DataFrame(
    [
        {
            "metric": "Latest TTM revenue",
            "value": latest_ttm_metrics["ttmRevenue"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM gross profit",
            "value": latest_ttm_metrics["ttmGrossProfit"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM operating income",
            "value": latest_ttm_metrics["ttmOperatingIncome"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM pre-tax income",
            "value": latest_ttm_metrics["ttmIncomeBeforeTax"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM net income",
            "value": latest_ttm_metrics["ttmNetIncome"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM gross margin",
            "value": latest_ttm_metrics["ttmGrossMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM operating margin",
            "value": latest_ttm_metrics["ttmOperatingMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM pre-tax margin",
            "value": latest_ttm_metrics["ttmIncomeBeforeTaxMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM net margin",
            "value": latest_ttm_metrics["ttmNetMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
    ]
)
ttm_summary_display = ttm_summary.copy()
ttm_summary_display["value"] = [
    format_ttm_summary_value(metric, value)
    for metric, value in zip(ttm_summary_display["metric"], ttm_summary_display["value"])
]
display(ttm_summary_display)

from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_ttm_profit_conversion

ttm_profit_fig = plot_ttm_profit_conversion(ttm_frame, analysis_label=analysis_label)
ttm_profit_fig.show(config={"responsive": True, "displaylogo": False})

## Margin Driver Analysis

These cells decompose profitability into gross margin, operating margin, and net margin drivers. Use them to compare revenue growth against cost structure changes and to see how below-the-line items affect final net margins.

In [ ]:
# 10. Plot gross margin drivers
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_gross_margin_drivers

gross_margin_fig = plot_gross_margin_drivers(
    annual_revenue,
    quarterly_revenue,
    chart_label=chart_label,
)
gross_margin_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 11. Plot operating margin drivers
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_operating_margin_drivers

operating_margin_fig = plot_operating_margin_drivers(
    annual_revenue,
    quarterly_revenue,
    chart_label=chart_label,
)
operating_margin_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 12. Plot net margin drivers
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_net_margin_drivers

net_margin_fig = plot_net_margin_drivers(
    annual_revenue,
    quarterly_revenue,
    chart_label=chart_label,
)
net_margin_fig.show(config={"responsive": True, "displaylogo": False})

## Revenue Segmentation

This section loads any non-plan-gated annual revenue segmentation returned by FMP and visualizes the available product and geographic breakdowns. If segmentation data is unavailable for the current ticker or plan, the downstream cells will show that through the loaded availability checks.

In [ ]:
# 13. Review annual revenue segmentation availability
available_revenue_segmentations

In [ ]:
# 14. Plot annual revenue segmentation
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_revenue_segmentation

revenue_segmentation_fig = plot_revenue_segmentation(
    annual_product_segmentation=annual_product_segmentation,
    annual_product_segments=annual_product_segments,
    annual_geographic_segmentation=annual_geographic_segmentation,
    annual_geographic_segments=annual_geographic_segments,
    chart_label=chart_label,
    symbol=SYMBOL,
)
revenue_segmentation_fig.show(config={"responsive": True, "displaylogo": False})